# Project Canary — Day 35 Average-Weight Forecast

**Purpose:** reproduce the complete weight-model workflow in a form the capstone team can run and defend.

**Business question:** given the weights recorded so far, what average building weight should we expect on Day 35, compared with the 1,800 g milestone?

## 1. Define Y, X, and the unit of analysis

- **Y target:** remaining growth = observed Day 35 bodyweight − current checkpoint bodyweight.
- **Final output:** current measured weight + predicted remaining gain.
- **One independent outcome:** one building in one cycle with a Day 35 measurement.
- **Training rows:** up to four checkpoint views of that outcome—Day 7, 14, 21, and 28. These are repeated views, not 124 independent flocks.
- **Candidate X inputs:** measurement day; latest/checkpoint weights; weight ÷ the farm target for that day; recent and cumulative average daily gain; current survival; and environmental-band exposure known by that review date.
- The interpolated 1,800 g target curve is an input/reference. It is **not** the Y label and does not manufacture an actual Day 35 result.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "FARM HARVEST DATA.xlsx"
MODEL_READY_DIR = ROOT / "outputs" / "model_ready"

from canary import load_workbook

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
dataset = load_workbook(DATA_PATH)
print(f"Source: {dataset.source_name}")
print(f"Canonical building-day rows: {len(dataset.daily):,}")
print(f"Recorded building-cycles: {len(dataset.cycles):,}")
print(f"Blocking data-quality checks passed: {dataset.quality.passed}")
print(f"Non-blocking warnings: {len(dataset.quality.warnings)}")

Source: FARM HARVEST DATA.xlsx
Canonical building-day rows: 1,666
Recorded building-cycles: 34
Blocking data-quality checks passed: True
Non-blocking warnings: 3


In [2]:
from canary import build_day35_feature_rows, build_day35_training_rows, train_day35_weight_baseline

rows = build_day35_training_rows(dataset)
engineered_rows = build_day35_feature_rows(dataset)
outcomes = rows[["cycle_id", "building_id", "actual_day35_weight_kg"]].drop_duplicates()
coverage = pd.DataFrame({
    "Measure": ["Historical cycles", "Distinct Day 35 building outcomes", "Checkpoint training rows", "At/above 1,800 g", "Below 1,800 g"],
    "Count": [rows["cycle_id"].nunique(), len(outcomes), len(rows), outcomes["actual_day35_weight_kg"].ge(1.8).sum(), outcomes["actual_day35_weight_kg"].lt(1.8).sum()],
})
coverage

In [3]:
exported = pd.read_csv(MODEL_READY_DIR / "day35_weight_training.csv")
assert len(exported) == len(engineered_rows) == 124
shared = [column for column in engineered_rows.columns if column in exported.columns]
left = engineered_rows[shared].copy().sort_values(["cycle_id", "building_id", "measurement_day"]).reset_index(drop=True)
right = exported[shared].copy().sort_values(["cycle_id", "building_id", "measurement_day"]).reset_index(drop=True)
for column in left.columns:
    if pd.api.types.is_numeric_dtype(left[column]):
        assert np.allclose(left[column], pd.to_numeric(right[column]), equal_nan=True)
    else:
        assert left[column].astype(str).equals(right[column].astype(str))
print("Export reconciliation passed: the CSV contains the exact 124 engineered weight rows.")

Export reconciliation passed: the CSV contains the exact 124 engineered weight rows.


## 2. Preprocessing and validation

1. Correct weight rows during workbook standardization and aggregate zone records to one building-day.
2. Keep only observed checkpoint weights and observed Day 35 labels; never fill a missing Day 35 Y from the target curve.
3. At each checkpoint, hide future checkpoint weights.
4. Give each building-cycle equal total weight across its repeated checkpoints.
5. Compare exactly five candidates: historical remaining-gain baseline, checkpoint linear regression, Ridge regression, robust Huber regression, and constrained Gradient Boosting.
6. Use **nested leave-one-complete-cycle-out cross-validation** so tuning, imputation and scaling never see the outer test cycle.
7. Optimize **cycle-macro MAE in kilograms**. A learned model replaces historical remaining gain only if it improves MAE by 10%, keeps positive R², places at least 70% within 200 g, remains stable, and improves target-side classification.

In [4]:
manifest = train_day35_weight_baseline(dataset)
print("Operational method:", manifest["selected_model"])
print("Best learned challenger:", manifest["research_champion"])
print("Champion gates:", manifest["champion_gates"])
print("Model version:", manifest["model_version"])
print("Selected X inputs:")
for feature in manifest["feature_columns"]:
    print(" -", feature)

Operational method: historical_remaining_gain
Best learned challenger: checkpoint_linear_remaining_gain
Champion gates: {'baseline': 'historical_remaining_gain', 'baseline_improvement_pct': -13.883167588070389, 'requires_at_least_10pct_mae_improvement': False, 'requires_positive_r2': False, 'requires_at_least_70pct_within_200g': False, 'requires_stable_worst_cycle': False, 'regression_gate_passed': False, 'requires_better_than_majority_target_side_accuracy': True, 'requires_recall_for_both_target_sides': True, 'target_classification_gate_passed': True, 'operational_fallback_applied': True}
Model version: day35-weight-2.0.0
Selected X inputs:
 - current_weight_kg
 - measurement_day


## 3. Candidate comparison

In [5]:
comparison = pd.DataFrame([
    {
        "Candidate": entry["model"],
        "Available": entry["available"],
        "Role": "Operational fallback" if entry["model"] == manifest["selected_model"] else "Best learned challenger" if entry["model"] == manifest["research_champion"] else "Compared",
        "MAE (g)": manifest["candidate_metrics"].get(entry["model"], {}).get("mae_kg", np.nan) * 1000,
        "Cycle-macro MAE (g)": manifest["candidate_metrics"].get(entry["model"], {}).get("cycle_macro_mae_kg", np.nan) * 1000,
        "RMSE (g)": manifest["candidate_metrics"].get(entry["model"], {}).get("rmse_kg", np.nan) * 1000,
        "R²": manifest["candidate_metrics"].get(entry["model"], {}).get("r2", np.nan),
        "Within 200 g": manifest["candidate_metrics"].get(entry["model"], {}).get("within_200g_rate", np.nan),
        "Target-side accuracy": manifest["candidate_metrics"].get(entry["model"], {}).get("target_side_accuracy", np.nan),
    }
    for entry in manifest["candidate_registry"]
]).sort_values("Cycle-macro MAE (g)")
comparison.round({"MAE (g)": 0, "Cycle-macro MAE (g)": 0, "RMSE (g)": 0, "R²": 3, "Bias (g)": 0, "Within 200 g": 3, "Target-side accuracy": 3})

In [6]:
cycle_performance = pd.DataFrame.from_dict(manifest["selected_metrics"]["cycle"], orient="index")
cycle_performance.index.name = "Held-out cycle"
cycle_performance.assign(
    mae_g=cycle_performance.mae_kg * 1000,
    rmse_g=cycle_performance.rmse_kg * 1000,
    bias_g=cycle_performance.bias_kg * 1000,
)[["rows", "mae_g", "rmse_g", "bias_g"]].round(2)

In [7]:
selected = manifest["selected_metrics"]
print(f"Selected held-out MAE: {selected['mae_kg']*1000:.0f} g")
print(f"Selected held-out RMSE: {selected['rmse_kg']*1000:.0f} g")
print(f"Held-out bias: {selected['bias_kg']*1000:+.0f} g")
print(f"Within 200 g: {selected['within_200g_rate']:.1%}")
print(f"Correct side of 1,800 g: {selected['target_side_accuracy']:.1%}")
print(f"Historical target hits: {manifest['actual_target_hits']} of {manifest['training_building_cycles']}")

Selected held-out MAE: 178 g
Selected held-out RMSE: 242 g
Held-out bias: +3 g
Within 200 g: 65.3%
Correct side of 1,800 g: 84.7%
Historical target hits: 5 of 31


**Interpretation:** no learned challenger cleared all approved gates, so historical remaining gain stays operational. The high target-side percentage partly reflects that 26 of 31 historical outcomes are below 1,800 g; it is not proof of balanced target classification.

## 4. What the selected model relies on

In [8]:
importance = pd.DataFrame(manifest["research_champion_permutation_importance"])
importance.head(10).rename(columns={
    "feature": "Input",
    "mean_mae_increase_kg": "Held-out MAE increase (kg)",
    "relative_importance_pct": "Relative held-out reliance (%)",
}).round(4)

These are held-out permutation importances for the best learned challenger, not causal effects. Weight features are correlated; use the operational forecast and recorded evidence rather than treating importance as an intervention instruction.

## 5. Day 14 held-out proof and one complete example

In [9]:
def cycle_bootstrap_mae(frame, error_column, repeats=5000, seed=42):
    # Bootstrap whole cycles, never individual rows, to preserve grouped evidence.
    rng = np.random.default_rng(seed)
    grouped = {cycle: group for cycle, group in frame.groupby("cycle_id")}
    cycles = np.array(list(grouped))
    estimates = []
    for _ in range(repeats):
        selected = rng.choice(cycles, size=len(cycles), replace=True)
        errors = np.concatenate([grouped[cycle][error_column].to_numpy(float) for cycle in selected])
        estimates.append(np.mean(np.abs(errors)))
    return np.quantile(estimates, [0.025, 0.975])

In [10]:
day14 = pd.DataFrame(manifest["day14_backtest"])
day14["error_g"] = day14["error_kg"] * 1000
ci = cycle_bootstrap_mae(day14, "error_g")
metrics = manifest["day14_backtest_metrics"]
print(f"Day 14 building outcomes: {metrics['building_cycles']}")
print(f"Day 14 MAE: {metrics['mae_kg']*1000:.0f} g")
print(f"Cycle-bootstrap 95% interval for Day 14 MAE: {ci[0]:.0f} to {ci[1]:.0f} g")
example = day14.iloc[0]
print("\nExample")
print(f"Cycle/building: {example.cycle_id} / {example.building_id}")
print(f"Day 14 measured weight: {example.current_weight_kg*1000:.0f} g")
print(f"Projected Day 35 weight: {example.predicted_day35_weight_kg*1000:.0f} g")
print(f"Recorded Day 35 weight: {example.actual_day35_weight_kg*1000:.0f} g")
print(f"Error = projected - recorded: {example.error_g:+.0f} g")
day14.head(8)[["cycle_id", "building_id", "current_weight_kg", "predicted_day35_weight_kg", "actual_day35_weight_kg", "error_g"]]

Day 14 building outcomes: 31
Day 14 MAE: 181 g
Cycle-bootstrap 95% interval for Day 14 MAE: 124 to 246 g

Example
Cycle/building: 2025-2 / Tags 1
Day 14 measured weight: 338 g
Projected Day 35 weight: 1573 g
Recorded Day 35 weight: 1810 g
Error = projected - recorded: -237 g


## 6. Historical remaining gain—why it remains in the comparison

For every eligible training building at a checkpoint age:

`remaining gain = observed Day 35 weight − checkpoint weight`

The baseline averages those gains in the training cycles and adds the average to the current weight. During validation, the held-out cycle is excluded. It remains the live operational method because the learned challengers did not clear every gate.

In [11]:
remaining = pd.DataFrame({
    "Checkpoint day": [int(day) for day in manifest["remaining_gain_by_measurement_day_kg"] if int(day) < 35],
    "Average historical remaining gain (g)": [gain * 1000 for day, gain in manifest["remaining_gain_by_measurement_day_kg"].items() if int(day) < 35],
}).sort_values("Checkpoint day")
remaining.round(0)

## 7. Why SMOTE or oversampling is not used

- This is regression, and standard SMOTE is a classification method.
- The 124 checkpoint rows come from only 31 independent building outcomes. Duplicating or synthesizing rows would not create new flocks.
- Synthetic weight paths may violate biological growth and can make validation look falsely precise.
- The class-like target imbalance is reported explicitly instead of hidden.

**Safer strategy:** regularized linear models, simple baselines, nested complete-cycle holdouts, checkpoint/horizon metrics, cycle-level bootstrap intervals, and more standardized Day 35 outcomes over time. A hierarchical model can be considered later, after more cycles—not as a capstone requirement.

## 8. Defense takeaway

Canary's weight output uses **historical remaining gain as the operational fallback**, validated on 31 historical Day 35 building outcomes and their earlier checkpoints. Learned linear and boosted challengers were tested under nested whole-cycle validation but did not clear all replacement gates. This is useful directional decision support, not a guarantee that a building will hit 1,800 g.